# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Functio to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [7]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': [
        'customers',
        'orders',
        'order_items',
        'products',
        'categories',
        'payments',
        'employees',
        'suppliers'
    ],
    'definition': [
        'Stores customer personal information including name, email, and contact details',
        'Records each purchase transaction made by a customer, including order date and status',
        'Line-level detail of each order, linking products and quantities to a parent order',
        'Contains product catalog data such as name, price, SKU, and stock quantity',
        'Groups products into logical categories for browsing and filtering',
        'Tracks payment transactions associated with orders, including method and amount',
        'Holds staff records including role, department, hire date, and manager',
        'Maintains supplier information for procurement and inventory management'
    ]
}

df = pd.DataFrame(data)
print(df)


         table                                         definition
0    customers  Stores customer personal information including...
1       orders  Records each purchase transaction made by a cu...
2  order_items  Line-level detail of each order, linking produ...
3     products  Contains product catalog data such as name, pr...
4   categories  Groups products into logical categories for br...
5     payments  Tracks payment transactions associated with or...
6    employees  Holds staff records including role, department...
7    suppliers  Maintains supplier information for procurement...


In [8]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])

In [9]:
print(text_tables)

customers: Stores customer personal information including name, email, and contact details
orders: Records each purchase transaction made by a customer, including order date and status
order_items: Line-level detail of each order, linking products and quantities to a parent order
products: Contains product catalog data such as name, price, SKU, and stock quantity
categories: Groups products into logical categories for browsing and filtering
payments: Tracks payment transactions associated with orders, including method and amount
employees: Holds staff records including role, department, hire date, and manager
suppliers: Maintains supplier information for procurement and inventory management


In [11]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""


In [13]:
#Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="What is the average order processing time by employee?")

In [15]:
print(return_OAI(pqt1))

```json
{
    "tables": ["orders", "employees"]
}
```


In [25]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                     question="Who are the customers?")

In [26]:
print(return_OAI(pqt3))

```json
{
    "tables": ["customers"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

1st Version

In [27]:
# Table and definitions sample
data_1 = {
    'table': [
        'car_models',
        'car_inventory',
        'age_groups',
        'car_colors',
        'top_speeds',
        'payments',
        'brands',
        'horsepower_specs'
    ],
    'definition': [
        'Contains details about each car model including model name, year, and associated brand',
        'Tracks the total count of cars available, sold, or in stock per model or category',
        'Defines age group segments (e.g. 18-25, 26-35) used to classify customers or target demographics',
        'Stores available color options for each car model, including color codes and names',
        'Records the maximum speed capability for each car model, measured in km/h or mph',
        'Tracks payment transactions for car purchases, including method, amount, and status',
        'Holds manufacturer/brand information such as brand name, country of origin, and founding year',
        'Stores engine horsepower ratings for each car model, used for performance comparisons'
    ]
}

df_1 = pd.DataFrame(data_1)
print(df_1)

              table                                         definition
0        car_models  Contains details about each car model includin...
1     car_inventory  Tracks the total count of cars available, sold...
2        age_groups  Defines age group segments (e.g. 18-25, 26-35)...
3        car_colors  Stores available color options for each car mo...
4        top_speeds  Records the maximum speed capability for each ...
5          payments  Tracks payment transactions for car purchases,...
6            brands  Holds manufacturer/brand information such as b...
7  horsepower_specs  Stores engine horsepower ratings for each car ...


In [32]:
text_tables_1 = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df_1.iterrows()])

In [40]:
print(text_tables_1)

car_models: Contains details about each car model including model name, year, and associated brand
car_inventory: Tracks the total count of cars available, sold, or in stock per model or category
age_groups: Defines age group segments (e.g. 18-25, 26-35) used to classify customers or target demographics
car_colors: Stores available color options for each car model, including color codes and names
top_speeds: Records the maximum speed capability for each car model, measured in km/h or mph
payments: Tracks payment transactions for car purchases, including method, amount, and status
brands: Holds manufacturer/brand information such as brand name, country of origin, and founding year
horsepower_specs: Stores engine horsepower ratings for each car model, used for performance comparisons


In [35]:
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Questyion:
{question}
"""

In [41]:
pqt100 = prompt_question_tables.format(tables=text_tables_1, question="What is the car model?")

In [42]:
print(return_OAI(pqt100))

{
    "tables": ["car_models"]
}


Test unrelated question

In [43]:
pqt101 = prompt_question_tables.format(tables=text_tables_1, question="Who is Donald Trump?")

In [44]:
print(return_OAI(pqt101))

```json
[]
```


In [45]:
pqt102 = prompt_question_tables.format(tables=text_tables_1, question="What are the age groups of the customer?")

In [46]:
print(return_OAI(pqt102))

```json
{
    "tables": ["age_groups"]
}
```


Report

- For direct, schema-aligned questions the model selected the obvious table(s). Example: "What is the car model?" → model picked car_models.
- For demographic/segmentation questions the model commonly selected age_groups appropriately when that table explicitly matched the question.
- For irrelevant questions (example in the notebook: "Who is Donald Trump?") the model tended to ignore the provided tables and returned general knowledge or an answer unrelated to the schema instead of returning an empty JSON or indicating no matching table.
- The model sometimes returned extra tables or missed necessary join-related tables when the definitions omitted relationship/column details (e.g., it might pick car_models but omit brands when the brand relationship is needed).
- When definitions are sparse, the model implicitly assumes relevant columns exist (it may behave as if fields like model_name, price, order_date, or processing_time exist), which can be a practical mismatch against a real schema.
- The model occasionally hallucinated schema details or implied columns/tables not present in the provided definitions.

Learned

- Provide minimal but explicit schema hints: include key column names and primary join columns (e.g., car_models(id, model_name, brand_id), brands(id, name)), not just high-level descriptions.
- Require the model to justify selections: ask for JSON plus a one-sentence rationale for each chosen table to reveal reasoning and catch dubious picks.
- Add explicit constraints in the prompt: e.g., "If none of the tables apply, return []" and "Do not include external knowledge — only use the tables listed."